# Practice 098 — Potential Outcomes & Randomized Experiments

**Theoretical context**: see `CLAUDE.md` in this folder before starting.

**Phases**: this notebook mirrors the phases in `CLAUDE.md` § Instructions.
Each phase's exercise calls into a `src/_0N_<phase_name>.py` companion module —
read that module's `TODO(human)` block before implementing it there, then
re-run the corresponding cell below.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import xy.pyplot as plt

from src.datasets import load_dataset
from src.plotting import cuped_variance_plot, power_curve_plot, randomization_distribution_plot

## Phase 1 — The science table & difference-in-means

We simulate a randomized experiment with a **known** true ATE (see `src/datasets.py`).
Because the data is synthetic, we can look at the full science table — both potential
outcomes `y0`/`y1` for every unit — something a real experiment never lets you see.

In [ ]:
data = load_dataset(n=500, tau=2.0, seed=0)
print(f"True ATE: {data.tau_true}")
data.science_table().head()

### Exercise — `src/_01_diff_in_means.py :: difference_in_means`

Open `src/_01_diff_in_means.py`, read the `TODO(human)` block above the function,
implement it there (not in this cell), then re-run the cell below.

In [ ]:
from src._01_diff_in_means import difference_in_means

ate_hat = difference_in_means(data.y_observed, data.treatment)
print(f"True ATE:            {data.tau_true:.4f}")
print(f"Difference-in-means: {ate_hat:.4f}")

## Phase 2 — Neyman's variance estimator

The diff-in-means estimator is a random variable — it depends on which units happened
to be drawn into the treated group. Neyman's design-based variance formula gives a
conservative estimate of that randomness, with no assumption on the outcome's
distribution.

### Exercise — `src/_02_neyman_variance.py :: neyman_variance`

Open `src/_02_neyman_variance.py`, read the `TODO(human)` block above the
function, implement it there, then re-run the cell below.

In [ ]:
from src._02_neyman_variance import neyman_variance

var_hat = neyman_variance(data.y_observed, data.treatment)
se_hat = np.sqrt(var_hat)
lo, hi = ate_hat - 1.96 * se_hat, ate_hat + 1.96 * se_hat
print(f"Neyman SE: {se_hat:.4f}")
print(f"95% CI:    [{lo:.4f}, {hi:.4f}]  (true ATE = {data.tau_true})")

## Phase 3 — Fisher's sharp null via randomization inference

Instead of a variance formula, Fisher's approach reshuffles the treatment labels
among the same fixed outcomes many times under the sharp null (`H0: no unit's outcome
would have changed`), building an exact null distribution for the diff-in-means
statistic.

### Exercise — `src/_03_randomization_inference.py :: randomization_test`

Open `src/_03_randomization_inference.py`, read the `TODO(human)` block above
the function, implement it there, then re-run the cell below.

In [ ]:
from src._03_randomization_inference import randomization_test

rng = np.random.default_rng(1)
observed, null_stats, p_value = randomization_test(data.y_observed, data.treatment, n_perm=2000, rng=rng)
print(f"Observed statistic: {observed:.4f}")
print(f"Two-sided p-value:  {p_value:.4f}")

fig = randomization_distribution_plot(null_stats, observed)
fig

## Phase 4 — Power analysis & minimum detectable effect

Before running an experiment, the practical question is "how many units do I need to
reliably detect an effect of the size I care about." We compute the minimum detectable
effect (MDE) across a grid of sample sizes, and the achieved power for the true ATE at
one candidate sample size.

### Exercise — `src/_04_power_analysis.py :: minimum_detectable_effect`, `power_for_effect`

Open `src/_04_power_analysis.py`, read the `TODO(human)` blocks above each
function, implement them there, then re-run the cell below.

In [ ]:
from src._04_power_analysis import minimum_detectable_effect, power_for_effect

sigma2 = float(np.var(data.y_observed, ddof=1))
n_grid = np.array([25, 50, 100, 150, 250, 400, 600, 900, 1300])
power_grid = np.array([power_for_effect(data.tau_true, n=int(n), sigma2=sigma2) for n in n_grid])

for n in (50, 250, 1000):
    print(f"n={n:5d}  MDE={minimum_detectable_effect(n, sigma2):.4f}")

fig = power_curve_plot(n_grid, power_grid)
fig

## Phase 5 — CUPED variance reduction

`x` is a pre-period covariate correlated with the outcome (see `src/datasets.py`).
CUPED adjusts the outcome using `x` to shrink the estimator's variance without
moving the point estimate.

### Exercise — `src/_05_cuped.py :: cuped_adjust`

Open `src/_05_cuped.py`, read the `TODO(human)` block above the function,
implement it there, then re-run the cell below.

In [ ]:
from src._05_cuped import cuped_adjust

y_adj, theta = cuped_adjust(data.y_observed, data.x)
ate_cuped = difference_in_means(y_adj, data.treatment)
var_raw = neyman_variance(data.y_observed, data.treatment)
var_cuped = neyman_variance(y_adj, data.treatment)

print(f"theta:                {theta:.4f}")
print(f"ATE (raw):            {ate_hat:.4f}   Neyman var: {var_raw:.4f}")
print(f"ATE (CUPED-adjusted): {ate_cuped:.4f}   Neyman var: {var_cuped:.4f}")
print(f"Variance reduction:   {(1 - var_cuped / var_raw):.1%}")

fig = cuped_variance_plot(var_raw, var_cuped)
fig

## Verification

Sanity-checks that must pass once every TODO is implemented.

In [ ]:
assert abs(ate_hat - data.tau_true) < 1.0, "diff-in-means should land reasonably close to the true ATE"
assert lo < data.tau_true < hi, "Neyman 95% CI should cover the true ATE"
assert null_stats.shape == (2000,)
assert 0.0 <= p_value <= 1.0
assert power_grid[-1] > power_grid[0], "power should increase with sample size"
assert var_cuped < var_raw, "CUPED should reduce variance given a correlated covariate"
print("OK")